Imports

In [ ]:
#import the whispermodel
from  faster_whisper import WhisperModel 
import re
from difflib import SequenceMatcher

Load the whisper model

In [ ]:
#load the model
model = WhisperModel(
                     'small',
                     device='cpu',
                     compute_type = 'int8'
                    ) 

Transription

In [ ]:
# transcribe the audio
audio_path = 'New recording 15.m4a_segment_4.wav'

def transribe(audio_path):
    segment, info = model.transcribe(audio_path,
                                    language  = 'ml'
                                    )    #during transcription , the data is transcribed as chuncks . and the output will contain transcription and the metadata -> which is  referred by variables segments and info . so when we print the transcriptions  - segment[0]+ segment[1].....+ ...segment[n] .which is print using a for loop or join this segments into a single string .
    
    #Storing the transribed text into a variable
    
    spoken_text = " ".join(seg.text  for seg in segment)


    # detected_language = info.language
    # print("detected language is : ",detected_language)

    # print('Transcription')
    # for i in segment:
    #     print(i.text)
    
    return {
        "text": spoken_text,
        "language" : info.language,
        "duration" : info.duration
    }

Core analysis engine for the Malayalam Reading Fluency Analyzer.
 
Takes:
  - expected_text : the paragraph the child was supposed to read
  - spoken_text   : what Whisper actually transcribed
  - duration_sec  : total audio duration from transcriber (for WPM)
 
Returns a structured report dict with:
  - accuracy %
  - WPM
  - missing words
  - extra words
  - substituted words
  - word-level diff (for highlighting in UI)

Normalize the Text

In [ ]:
def normalize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


Reading analyzer

In [ ]:
def analyzer(
    expected:str, 
    spoken : str, 
    duration :float):
    
    expected_text = normalize(expected)
    spoken_text = normalize(spoken)
    
    #split the expected and spoken sentences into words.
    
    expected_words = expected_text.split()
    spoken_words = spoken_text.split()
    
    
    matcher = SequenceMatcher(
        None,
        expected_words,
        spoken_words
    )
    
    
    missing = []
    extra = []
    substituted = []
    correct = []
    
    
    
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():

        if tag == "equal":
            correct.extend(expected_words[i1:i2])

        elif tag == "delete":
            missing.extend(expected_words[i1:i2])

        elif tag == "insert":
            extra.extend(spoken_words[j1:j2])

        elif tag == "replace":

            substituted.append({
                "expected": expected_words[i1:i2],
                "spoken": spoken_words[j1:j2]
            })

    
    
    
    total_expected = len(expected_words)
    
    accuracy = (
        len(correct) / total_expected * 100
        if total_expected else 0
    )
    
    
    minutes = duration/60
    wpm = len(spoken_words)/minutes
    
    
    report = {

        "accuracy": round(accuracy,2),

        "wpm": round(wpm,2),

        "correct_words": correct,

        "missing_words": missing,

        "extra_words": extra,

        "substituted_words": substituted
    }
    
    
    return report
    
    
    
    

The report

In [ ]:
expected = """
അവൾ പുസ്തകം വായിക്കുന്നു
"""

audio = transribe(audio_path)

report = analyzer(
    expected,
    audio["text"],
    audio["duration"]
)

for key,value in report.items():
    print(f"{key} : {value}")

accuracy : 0.0
wpm : 67.64
correct_words : []
missing_words : []
extra_words : []
substituted_words : [{'expected': ['അവൾ', 'പസതക', 'വയകകനന'], 'spoken': ['awal', 'pustagam', 'vayikunnu']}]
